<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 1 OPTIONAL · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Lab 1 Optional: Connect Metabase and Build a Dashboard</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Run Metabase beside the Lab 1 Doris sandbox, install the Apache Doris community driver, connect to the <code>events</code> table, and publish a daily-revenue chart to a dashboard.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Optional · Metabase 0.59.6.3 · Metabase Apache Doris Driver 1.0.0 · Docker · Dashboard</span>
</div>

This extension is optional and takes approximately 20–30 minutes. It demonstrates how Doris serves as the analytical database behind a BI application; it is not required to complete Lab 1.


### Initialize the Lab 1 Optional

Run the next cell once before continuing. It loads the same local styling and command runner used by Lab 1. It does not download software, start containers, or change Doris data. Run it again after restarting the Jupyter kernel.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(
    lab_dir=COURSE_ROOT,
    ready_title="Lab 1 Optional tools are ready",
    ready_message="Continue to Section 1 and run the remaining cells in order.",
);


## 1. Understand the integration

Metabase is an open-source business intelligence application. The Metabase Apache Doris Driver is a standalone community driver that lets Metabase discover Doris metadata and send SQL to FE port `9030` through the MySQL-compatible protocol.

| Component | Role in this optional lab |
|---|---|
| Browser | Opens the Metabase web interface on host port `3000`. |
| Metabase container | Stores questions and dashboards, loads the Doris driver, and sends SQL. |
| Docker network `doris-course` | Resolves the Doris container by the hostname `doris`. |
| Frontend (FE) | Accepts and analyzes SQL on port `9030`, creates and optimizes a distributed query plan, and assigns its plan fragments. |
| Backend (BE) | Executes the assigned plan fragments to scan and aggregate the Lab 1 `events` table. |

The Metabase container uses the named volume `metabase-course-data`, so its application database survives a container stop. The Doris table remains in the separate `doris-fe-meta` and `doris-be-storage` volumes created by Lab 1.

<div style="max-width:920px;border:1px solid #dbe4e8;border-left:3px solid #d97706;border-radius:4px;background:#fffbeb;color:#334155;padding:10px 12px;margin:14px 0"><strong style="color:#17212b;display:block;margin-bottom:2px">Third-party driver</strong>The Metabase Apache Doris Driver is maintained outside the Apache Doris project. The notebook installs version 1.0.0. Evaluate its security, license, and compatibility separately before production use.</div>


## 2. Prepare the Metabase environment

Complete Lab 1 first. The next cell expects the healthy `doris` container, the `doris-course` Docker network, and `doris_course.events` containing 10,158,080 rows.

<div style="max-width:100%;overflow-x:auto;margin:16px 0 20px;padding-bottom:4px">
<table style="border-collapse:separate;border-spacing:6px 0;min-width:980px;width:100%;table-layout:fixed;margin:0">
<tr>
<td style="width:165px;vertical-align:top;border:1px solid #dbe4e8;border-top:3px solid #0f766e;border-radius:4px;padding:10px;background:#f8fbfb"><strong>1 · Verify Doris</strong><br><span style="color:#64748b;font-size:12px;line-height:1.45">Check the container, network, and baseline table</span></td>
<td style="width:24px;text-align:center;vertical-align:middle;border:0;color:#94a3b8;font-size:22px;padding:0">&#8594;</td>
<td style="width:165px;vertical-align:top;border:1px solid #dbe4e8;border-top:3px solid #0f766e;border-radius:4px;padding:10px;background:#f8fbfb"><strong>2 · Download driver</strong><br><span style="color:#64748b;font-size:12px;line-height:1.45">Fetch the pinned community-driver JAR</span></td>
<td style="width:24px;text-align:center;vertical-align:middle;border:0;color:#94a3b8;font-size:22px;padding:0">&#8594;</td>
<td style="width:165px;vertical-align:top;border:1px solid #dbe4e8;border-top:3px solid #0f766e;border-radius:4px;padding:10px;background:#f8fbfb"><strong>3 · Create storage</strong><br><span style="color:#64748b;font-size:12px;line-height:1.45">Create or reuse the Metabase named volume</span></td>
<td style="width:24px;text-align:center;vertical-align:middle;border:0;color:#94a3b8;font-size:22px;padding:0">&#8594;</td>
<td style="width:165px;vertical-align:top;border:1px solid #dbe4e8;border-top:3px solid #0f766e;border-radius:4px;padding:10px;background:#f8fbfb"><strong>4 · Start Metabase</strong><br><span style="color:#64748b;font-size:12px;line-height:1.45">Pull the image and start or reuse the container</span></td>
<td style="width:24px;text-align:center;vertical-align:middle;border:0;color:#94a3b8;font-size:22px;padding:0">&#8594;</td>
<td style="width:165px;vertical-align:top;border:1px solid #dbe4e8;border-top:3px solid #0f766e;border-radius:4px;padding:10px;background:#f8fbfb"><strong>5 · Wait for readiness</strong><br><span style="color:#64748b;font-size:12px;line-height:1.45">Check the Metabase health API</span></td>
</tr>
</table>
</div>

The variables at the top of the Bash cell control all reusable resource names and versions. The same commands run on macOS Apple Silicon and Linux x86_64; Docker selects the native image architecture.


In [ ]:
lab.shell(r"""
set -euo pipefail

METABASE_IMAGE="metabase/metabase:v0.59.6.3"
METABASE_CONTAINER="metabase"
METABASE_VOLUME="metabase-course-data"
DORIS_CONTAINER="doris"
DORIS_NETWORK="doris-course"
DRIVER_VERSION="v1.0.0"
DRIVER_FILE="doris.metabase-driver-v1.0.0.jar"
DRIVER_URL="https://github.com/velodb/metabase-doris-driver/releases/download/${DRIVER_VERSION}/${DRIVER_FILE}"

LAB_DIR="$PWD"
if [ ! -f "$LAB_DIR/pyproject.toml" ]; then
  echo "Start JupyterLab from the 01-real-time-analytics course directory."
  exit 2
fi
ASSETS_DIR="$LAB_DIR/level1/module01-introduction/assets"
DRIVER_PATH="$ASSETS_DIR/$DRIVER_FILE"

echo "[1/5] Verify the Lab 1 Doris sandbox and baseline table"
if ! command -v docker >/dev/null 2>&1; then
  echo "Docker CLI was not found. Complete Lab 1 before running this optional lab."
  exit 3
fi
if ! docker info >/dev/null 2>&1; then
  echo "Docker is not ready. Start Docker Desktop or Docker Engine, then rerun this cell."
  exit 4
fi
if ! docker container inspect "$DORIS_CONTAINER" >/dev/null 2>&1; then
  echo "The Lab 1 Doris container does not exist. Complete Lab 1 first."
  exit 5
fi
if [ "$(docker inspect --format '{{.State.Status}}' "$DORIS_CONTAINER")" != "running" ]; then
  docker start "$DORIS_CONTAINER"
fi
for attempt in $(seq 1 150); do
  DORIS_HEALTH=$(docker inspect --format '{{if .State.Health}}{{.State.Health.Status}}{{else}}none{{end}}' "$DORIS_CONTAINER")
  [ "$DORIS_HEALTH" = "healthy" ] && break
  [ "$attempt" -eq 150 ] && { echo "Doris did not become healthy."; exit 6; }
  sleep 2
done
if ! docker network inspect "$DORIS_NETWORK" >/dev/null 2>&1; then
  echo "The $DORIS_NETWORK network is missing. Rerun the Lab 1 environment preparation."
  exit 7
fi
EVENT_ROWS=$(docker exec "$DORIS_CONTAINER" mysql -uroot -h127.0.0.1 -P9030 -N \
  -e "SELECT COUNT(*) FROM doris_course.events")
if [ "$EVENT_ROWS" != "10158080" ]; then
  echo "Expected 10158080 rows in doris_course.events; found $EVENT_ROWS."
  exit 8
fi

echo "[2/5] Download the pinned Apache Doris community driver"
mkdir -p "$ASSETS_DIR"
if [ ! -f "$DRIVER_PATH" ]; then
  curl -fL --retry 3 --output "$DRIVER_PATH" "$DRIVER_URL"
else
  echo "Reusing $DRIVER_PATH"
fi

echo "[3/5] Create or reuse persistent Metabase storage"
docker volume inspect "$METABASE_VOLUME" >/dev/null 2>&1 || docker volume create "$METABASE_VOLUME"

echo "[4/5] Pull the pinned image and start or reuse Metabase"
docker pull "$METABASE_IMAGE"
if docker container inspect "$METABASE_CONTAINER" >/dev/null 2>&1; then
  CURRENT_IMAGE=$(docker inspect --format '{{.Config.Image}}' "$METABASE_CONTAINER")
  CURRENT_NETWORKS=$(docker inspect --format '{{range $name, $_ := .NetworkSettings.Networks}}{{println $name}}{{end}}' "$METABASE_CONTAINER")
  CURRENT_MOUNTS=$(docker inspect --format '{{range .Mounts}}{{printf "%s|%s\n" .Name .Destination}}{{end}}' "$METABASE_CONTAINER")
  CURRENT_PORTS=$(docker port "$METABASE_CONTAINER" 2>/dev/null || true)
  if [ "$CURRENT_IMAGE" != "$METABASE_IMAGE" ] \
     || ! grep -Fxq "$DORIS_NETWORK" <<<"$CURRENT_NETWORKS" \
     || ! grep -Fxq "$METABASE_VOLUME|/metabase-data" <<<"$CURRENT_MOUNTS" \
     || ! grep -Fq '|/plugins/doris.metabase-driver.jar' <<<"$CURRENT_MOUNTS" \
     || ! grep -Eq '3000/tcp.*127\.0\.0\.1:3000$' <<<"$CURRENT_PORTS"; then
    echo "The existing metabase container does not match this optional lab configuration."
    echo "It was left unchanged; rename it or choose different resource names above."
    exit 11
  fi
  if [ "$(docker inspect --format '{{.State.Status}}' "$METABASE_CONTAINER")" != "running" ]; then
    docker start "$METABASE_CONTAINER"
  fi
else
  if command -v lsof >/dev/null 2>&1 && lsof -nP -iTCP:3000 -sTCP:LISTEN >/dev/null 2>&1; then
    echo "Host port 3000 is already in use. Stop the conflicting service or change the port mapping."
    exit 12
  fi
  docker run -d \
    --name "$METABASE_CONTAINER" \
    --network "$DORIS_NETWORK" \
    -p 127.0.0.1:3000:3000 \
    -e MB_DB_FILE=/metabase-data/metabase.db \
    -v "$METABASE_VOLUME:/metabase-data" \
    -v "$DRIVER_PATH:/plugins/doris.metabase-driver.jar:ro" \
    "$METABASE_IMAGE"
fi

echo "[5/5] Wait for the Metabase health API"
for attempt in $(seq 1 150); do
  if HEALTH_RESPONSE=$(curl -fsS --max-time 3 http://127.0.0.1:3000/api/health 2>/dev/null); then
    echo "$HEALTH_RESPONSE"
    break
  fi
  if [ "$attempt" -eq 150 ]; then
    docker logs --tail 100 "$METABASE_CONTAINER"
    echo "Metabase did not become ready within 5 minutes."
    exit 13
  fi
  [ $((attempt % 10)) -eq 0 ] && echo "Waiting for Metabase (${attempt}/150)..."
  sleep 2
done
docker ps --filter "name=^/${METABASE_CONTAINER}$"
""", title="Prepare the Metabase environment");


**Expected result:** all five workflow items complete successfully. The health response contains `"status":"ok"`, and `docker ps` shows both the `metabase` and `doris` containers running. The downloaded driver remains in `assets/`, while Metabase application data is stored in the `metabase-course-data` named volume.

If the cell fails, expand **View diagnostic log**. A port conflict stops the workflow before a new Metabase container is created.


## 3. Open Metabase and add Doris

Open [http://127.0.0.1:3000](http://127.0.0.1:3000). On the first launch, complete the Metabase administrator setup. The administrator account belongs to Metabase; it is separate from the Doris user.

Navigate to **Admin Settings → Databases → Add database**, select **Apache Doris**, and enter:

| Field | Value |
|---|---|
| Display name | `Doris Course` |
| Host | `doris` |
| Port | `9030` |
| Catalog | `internal` |
| Database | `doris_course` |
| Username | `root` |
| Password | Leave blank for this disposable local sandbox only |
| SSL | Disabled |

> **Set Database to `doris_course`:** although Metabase labels this field as optional, leaving it empty enables multi-database discovery inside the `internal` Catalog. Metabase can then display tables from several visible databases, and the unqualified name `events` does not reliably identify `internal.doris_course.events`.

Use `doris`, not `127.0.0.1`, as the Host. Inside the Metabase container, `127.0.0.1` refers to Metabase itself; Docker DNS resolves `doris` to the Doris container on the shared `doris-course` network.

If the connection was saved without a Database, do not delete the Metabase container or metadata volume. Open **Admin Settings → Databases → Doris Course**, edit the connection, set **Database** to `doris_course`, and save it. Then run **Sync database schema now**. Until that correction is made, Native SQL can address the intended table explicitly as `internal.doris_course.events`.

**Expected result:** the connection test succeeds and synchronization is limited to `internal.doris_course`. Metabase discovers the course `events` table; metadata synchronization can continue briefly after the connection is saved.


## 4. Create a question and line chart

Select **New → SQL query**, choose **Doris Course**, and run:

```sql
SELECT
    TO_DATE(event_time) AS event_date,
    SUM(revenue) AS total_revenue
FROM events
WHERE event_type = 'purchase'
GROUP BY TO_DATE(event_time)
ORDER BY event_date;
```

The complete table name ensures that Native SQL reads `doris_course.events`, rather than the similarly named `information_schema.EVENTS` system view. `TO_DATE(event_time)` groups all purchases from the same calendar day.

Choose **Visualization → Line**, configure `event_date` as the X-axis and `total_revenue` as the Y-axis, and save the Question as **Daily Revenue** in the `Our analytics` Collection.

**Expected result:** the query returns one row per event date in chronological order, and the line chart plots daily purchase revenue without an additional series.


## 5. Publish the dashboard

The creation dialog defines the empty Dashboard and where Metabase stores it. Complete the fields shown in the dialog:

1. Select **+ New → Dashboard**.
2. In **Name**, enter `Doris Course Overview`.
3. In **Description**, enter `Daily purchase revenue from Apache Doris.`.
4. Under **Which collection should this go in?**, select `Our analytics`. 
5. Select **Create**.

Metabase now opens the new, empty Dashboard in edit mode. Add the saved Question from this screen:

6. Select the **+** button in the upper-right corner of the Dashboard.
7. Choose the option for an existing Question, then search for and select `Daily Revenue`.
8. Place the line-chart card on the Dashboard and resize it if needed.
9. Select **Save** in the upper-right corner to save the Dashboard layout.

**Expected result:** `Doris Course Overview` is stored in `Our analytics` and contains one card named `Daily Revenue`. The card displays the daily purchase-revenue line chart. Leaving and reopening the Dashboard preserves the card and its layout.


### Stop Metabase

Run this optional cell when you want to release the Metabase process and port 3000. It keeps the container and the `metabase-course-data` named volume, so the Doris connection, saved Question, and dashboard are not deleted. The Doris container and `events` table are unaffected.


In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop metabase
docker inspect --format 'container={{.State.Status}}' metabase
""", title="Stop Metabase");


**Expected result:** Docker reports `container=exited`; the Metabase named volume remains available.

### Restart Metabase

Run this cell when you want to reopen the saved dashboard. It restores the Doris dependency, starts or reuses the existing Metabase container, and waits until the Metabase health API reports that the application is ready.

This restart cell is idempotent: it can be run when Docker Desktop and the container are stopped, starting, or already running. On macOS it opens Docker Desktop when necessary; on Linux, start Docker Engine before running the cell.


In [ ]:
lab.start_container("doris")
lab.start_container("metabase", wait_for_healthy=False)

lab.shell(r"""
set -euo pipefail

for attempt in $(seq 1 150); do
  if HEALTH_RESPONSE=$(curl -fsS --max-time 3 http://127.0.0.1:3000/api/health 2>/dev/null); then
    echo "$HEALTH_RESPONSE"
    break
  fi

  if [ "$attempt" -eq 150 ]; then
    docker logs --tail 100 metabase
    exit 1
  fi

  if [ $((attempt % 10)) -eq 0 ]; then
    echo "Waiting for Metabase (${attempt}/150)..."
  fi
  sleep 2
done
""", title="Restart Metabase")


**Expected result:** the health response contains `"status":"ok"`. Reopen [http://127.0.0.1:3000](http://127.0.0.1:3000); the Doris connection, saved Question, and dashboard remain available.

### Common problems

| Symptom | Check |
|---|---|
| **Apache Doris** is absent from the database-type list | Run `docker logs metabase` and confirm `/plugins/doris.metabase-driver.jar` was registered. |
| Connection test fails | Confirm Host is `doris`, Port is `9030`, both containers use `doris-course`, and the Doris container is healthy. |
| `events` is not visible | Confirm Database is `doris_course`, then run **Sync database schema now** in Metabase. |
| `DESC events` shows `EVENT_CATALOG` and `EVENT_SCHEMA` | The unqualified name resolved to `information_schema.EVENTS`. Use `internal.doris_course.events` in Native SQL. |
| Browser cannot open port 3000 | Check `docker ps`, the health endpoint, and whether another host process owns port `3000`. |

## Lab 1 Optional complete

You installed the community driver, ran Metabase and Doris on one Docker network, connected Metabase to the Doris FE, executed an analytical query in Doris, and published the result as a persistent dashboard.

Official reference: [Apache Doris — Metabase integration](https://doris.apache.org/docs/4.x/connection-integration/data-integration/metabase/)
